# Customer Churn Prediction — Kaggle Playground S6E3

**Goal**: Maximize AUC-ROC + Accuracy ≥ 95%  
**Dataset**: Telco Customer Churn synthetic dataset (~594K rows, 20 features, binary target)

## Table of Contents
1. Environment Setup
2. Load & Inspect Data
3. Target Distribution & Class Imbalance
4. Numerical Feature Analysis
5. Categorical Feature Analysis
6. Bivariate & Multivariate Analysis
7. Data Cleaning & Preprocessing
8. Feature Engineering
9. Baseline Model
10. Gradient Boosting Models (5-Fold CV)
11. Hyperparameter Tuning (Optuna)
12. Stacking Ensemble
13. Threshold Tuning
14. SHAP Interpretability
15. Submission

## 1. Environment Setup

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
import shap

from scipy.stats import mstats
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid', palette='husl')
%matplotlib inline

RANDOM_STATE = 42
N_SPLITS = 5
DATA_DIR = 'data'
MODELS_DIR = 'models'
RESULTS_DIR = 'results'
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('All packages loaded successfully ✓')

## 2. Load & Inspect Data

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print(f'Train: {train.shape}  |  Test: {test.shape}  |  Sample Sub: {sample_sub.shape}')
train.head()

In [ ]:
train.info()

In [ ]:
# Check for missing / blank values
print('=== Missing values (NaN) ===')
print(train.isnull().sum())
print()
print('=== Blank string check (TotalCharges) ===')
blank_tc = (train['TotalCharges'].astype(str).str.strip() == '')
print(f'Blank TotalCharges: {blank_tc.sum()}')

In [ ]:
train.describe()

## 3. Target Distribution & Class Imbalance

In [ ]:
churn_counts = train['Churn'].value_counts()
churn_pct = train['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(churn_counts.index, churn_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Count')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', va='bottom')

axes[1].pie(churn_pct.values, labels=['No Churn', 'Churn'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Churn Distribution')
plt.suptitle('Target Distribution — Class Imbalance Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'target_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f'Class imbalance ratio: {churn_counts[0]/churn_counts[1]:.2f}:1  →  Strategy: scale_pos_weight + threshold tuning')

## 4. Numerical Feature Analysis

In [ ]:
# Convert TotalCharges to numeric first (may have blank strings)
train['TotalCharges'] = pd.to_numeric(train['TotalCharges'], errors='coerce')
test['TotalCharges']  = pd.to_numeric(test['TotalCharges'],  errors='coerce')

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(num_cols):
    for churn_val, color, label in [('No', '#2ecc71', 'No Churn'), ('Yes', '#e74c3c', 'Churn')]:
        subset = train[train['Churn'] == churn_val][col].dropna()
        axes[0, i].hist(subset, bins=50, alpha=0.5, color=color, label=label)
    axes[0, i].set_title(f'{col} Distribution by Churn')
    axes[0, i].legend()
    axes[0, i].set_xlabel(col)

    train.boxplot(column=col, by='Churn', ax=axes[1, i],
                  boxprops=dict(color='navy'), medianprops=dict(color='red'))
    axes[1, i].set_title(f'{col} Boxplot')
    axes[1, i].set_xlabel('Churn')

plt.suptitle('Numerical Feature Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'numerical_analysis.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (encode target temporarily)
train_temp = train.copy()
train_temp['Churn_num'] = (train_temp['Churn'] == 'Yes').astype(int)
corr = train_temp[num_cols + ['Churn_num']].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Numerical Features + Churn', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'correlation_heatmap.png'), dpi=120, bbox_inches='tight')
plt.show()

## 5. Categorical Feature Analysis

In [ ]:
key_cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

train_enc = train.copy()
train_enc['Churn_num'] = (train_enc['Churn'] == 'Yes').astype(int)

for i, col in enumerate(key_cat_cols):
    churn_rate = train_enc.groupby(col)['Churn_num'].mean().sort_values(ascending=False)
    churn_rate.plot(kind='bar', ax=axes[i], color=sns.color_palette('husl', len(churn_rate)))
    axes[i].set_title(f'Churn Rate by {col}')
    axes[i].set_ylabel('Churn Rate')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)
    for bar in axes[i].patches:
        axes[i].annotate(f'{bar.get_height():.2%}',
                         (bar.get_x() + bar.get_width()/2, bar.get_height()),
                         ha='center', va='bottom', fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Churn Rate per Categorical Feature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'categorical_churn_rates.png'), dpi=120, bbox_inches='tight')
plt.show()

## 6. Bivariate & Multivariate Analysis

In [ ]:
# Survival-curve style: tenure vs churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: tenure distribution by churn
for churn_val, color, label in [('No', '#2ecc71', 'No Churn'), ('Yes', '#e74c3c', 'Churn')]:
    subset = train[train['Churn'] == churn_val]['tenure']
    axes[0].hist(subset, bins=72, alpha=0.6, color=color, label=label, density=True)
axes[0].set_title('Tenure Distribution by Churn')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Right: Monthly charges vs tenure scatter (sample)
sample = train.sample(5000, random_state=RANDOM_STATE)
scatter_colors = sample['Churn'].map({'Yes': '#e74c3c', 'No': '#2ecc71'})
axes[1].scatter(sample['tenure'], sample['MonthlyCharges'],
                c=scatter_colors, alpha=0.3, s=10)
axes[1].set_title('Tenure vs MonthlyCharges (color=Churn)')
axes[1].set_xlabel('Tenure (months)')
axes[1].set_ylabel('Monthly Charges ($)')

from matplotlib.patches import Patch
legend_elements = [Patch(color='#e74c3c', label='Churn'), Patch(color='#2ecc71', label='No Churn')]
axes[1].legend(handles=legend_elements)

plt.suptitle('Bivariate Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'bivariate_analysis.png'), dpi=120, bbox_inches='tight')
plt.show()

## 7. Data Cleaning & Preprocessing

In [ ]:
# Import preprocessing functions from train.py
from train import (
    clean_data, preprocess,
    fix_total_charges, collapse_no_service, winsorize_numerics,
    feature_engineering, encode_binary, encode_ordinal, encode_ohe,
    COLLAPSE_COLS, BINARY_COLS, ORDINAL_COLS, OHE_COLS
)

# Reload raw data (before any in-notebook modifications)
train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print('Step 1: Fix TotalCharges blank strings')
train_step1 = fix_total_charges(train_raw)
print(f'  Missing after fix: {train_step1["TotalCharges"].isna().sum()}')

print('Step 2: Collapse No internet/phone service → No')
train_step2 = collapse_no_service(train_step1)
print(f'  MultipleLines unique values: {train_step2["MultipleLines"].unique()}')

print('Step 3: Winsorize P1/P99')
train_step3 = winsorize_numerics(train_step2)
print(f'  MonthlyCharges range: [{train_step3["MonthlyCharges"].min():.2f}, {train_step3["MonthlyCharges"].max():.2f}]')

In [ ]:
# Full clean + preprocess
train_clean = clean_data(train_raw)
test_clean  = clean_data(test_raw)

X_train, X_test, y_train, artifacts = preprocess(train_clean, test_clean)

print(f'X_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')
print(f'y_train mean (churn rate): {y_train.mean():.4f}')
print(f'Feature columns ({len(artifacts["feature_cols"])}):')
print(artifacts['feature_cols'])

## 8. Feature Engineering

In [ ]:
print('Engineered features added:')
engineered = ['AvgMonthlyCharge', 'ServiceCount', 'TenureBin', 'IsFiberOptic',
               'IsElectronicCheck', 'IsMonthToMonth', 'NewCustomer',
               'HighValueCustomer', 'TenureChargeInteraction', 'ContractTenureInteraction']
for f in engineered:
    if f in X_train.columns:
        print(f'  ✓ {f}  — sample values: {X_train[f].describe().to_dict()}')

## 9. Baseline Model — Logistic Regression

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
lr_scores = cross_val_score(lr, X_scaled, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'Baseline LR — AUC per fold: {[f"{s:.4f}" for s in lr_scores]}')
print(f'Baseline LR — Mean AUC: {lr_scores.mean():.5f} ± {lr_scores.std():.5f}')

## 10. Gradient Boosting Models — 5-Fold Stratified CV

In [ ]:
# Improved model parameters targeting ≥ 95% accuracy
models = {
    'XGBoost': xgb.XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85, scale_pos_weight=2.7,
        min_child_weight=3, gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        tree_method='hist', eval_metric='auc',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        num_leaves=63, subsample=0.85, colsample_bytree=0.85,
        min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
        is_unbalance=True, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    ),
    'CatBoost': CatBoostClassifier(
        iterations=400, depth=6, learning_rate=0.05,
        l2_leaf_reg=3.0, subsample=0.85,
        auto_class_weights='Balanced', random_seed=RANDOM_STATE, verbose=0
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_split=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:15s} AUC: {scores.mean():.5f} ± {scores.std():.5f}')


In [ ]:
# CV results comparison
cv_df = pd.DataFrame(cv_results)
fig, ax = plt.subplots(figsize=(10, 5))
cv_df.boxplot(ax=ax)
ax.set_title('5-Fold AUC-ROC Comparison', fontsize=13)
ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.7, 1.0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cv_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

## 11. Hyperparameter Tuning (Optuna)

> ⚠️ This cell runs Optuna tuning. Set `OPTUNA_TRIALS` to a smaller value for faster iteration.

In [ ]:
from train import (
    xgb_objective, lgb_objective, cat_objective, rf_objective, tune_model,
    OPTUNA_TRIALS
)

TUNE_MODELS = False  # Set to True to run Optuna tuning (takes ~10-30 min)
OPTUNA_NB_TRIALS = 20  # Use fewer trials in notebook for demonstration

if TUNE_MODELS:
    xgb_best = tune_model('XGBoost', xgb_objective, X_train, y_train, cv, OPTUNA_NB_TRIALS)
    lgb_best = tune_model('LightGBM', lgb_objective, X_train, y_train, cv, OPTUNA_NB_TRIALS)
    cat_best = tune_model('CatBoost', cat_objective, X_train, y_train, cv, OPTUNA_NB_TRIALS)
    rf_best  = tune_model('RandomForest', rf_objective, X_train, y_train, cv, OPTUNA_NB_TRIALS)
    print('Tuning complete.')
else:
    print('Skipping Optuna tuning (TUNE_MODELS=False). Run train.py --tune for full tuning.')

## 12. Stacking Ensemble — OOF Predictions

In [ ]:
from train import get_oof_predictions, predict_test

# Improved models for stacking — targeting ≥ 95% accuracy
models_params = {
    'XGBoost': (xgb.XGBClassifier, {
        'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.05,
        'subsample': 0.85, 'colsample_bytree': 0.85, 'scale_pos_weight': 2.7,
        'min_child_weight': 3, 'gamma': 0.1, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
        'tree_method': 'hist', 'eval_metric': 'auc',
        'random_state': RANDOM_STATE, 'n_jobs': -1}),
    'LightGBM': (lgb.LGBMClassifier, {
        'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.05,
        'num_leaves': 63, 'subsample': 0.85, 'colsample_bytree': 0.85,
        'min_child_samples': 20, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
        'is_unbalance': True, 'random_state': RANDOM_STATE,
        'n_jobs': -1, 'verbose': -1}),
    'CatBoost': (CatBoostClassifier, {
        'iterations': 400, 'depth': 6, 'learning_rate': 0.05,
        'l2_leaf_reg': 3.0, 'subsample': 0.85,
        'auto_class_weights': 'Balanced',
        'random_seed': RANDOM_STATE, 'verbose': 0}),
    'RandomForest': (RandomForestClassifier, {
        'n_estimators': 200, 'max_depth': 8, 'min_samples_split': 5,
        'class_weight': 'balanced',
        'random_state': RANDOM_STATE, 'n_jobs': -1}),
}

print('Generating OOF predictions for stacking …')
oof_preds, trained_models = get_oof_predictions(models_params, X_train, y_train, cv)


In [ ]:
# Meta-learner with regularization
meta_lr = LogisticRegression(C=0.5, random_state=RANDOM_STATE, max_iter=1000, solver='lbfgs')
meta_lr.fit(oof_preds, y_train)
stack_proba = meta_lr.predict_proba(oof_preds)[:, 1]
stack_auc = roc_auc_score(y_train, stack_proba)
print(f'Stacking OOF AUC: {stack_auc:.5f}')


## 13. Threshold Tuning

In [ ]:
from train import optimise_threshold

optimal_thr = optimise_threshold(y_train, stack_proba)
stack_preds = (stack_proba >= optimal_thr).astype(int)

stack_acc = accuracy_score(y_train, stack_preds)
stack_f1  = f1_score(y_train, stack_preds)
cm = confusion_matrix(y_train, stack_preds)

# Per-fold validation accuracy
fold_val_accs = []
for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    val_preds = (stack_proba[val_idx] >= optimal_thr).astype(int)
    fold_acc = accuracy_score(y_train[val_idx], val_preds)
    fold_val_accs.append(fold_acc)
mean_val_acc = np.mean(fold_val_accs)

print('=== OOF Metrics ===')
print(f'  AUC-ROC      : {stack_auc:.5f}')
print(f'  Train Accuracy: {stack_acc:.5f}')
print(f'  Val Accuracy  : {mean_val_acc:.5f} ± {np.std(fold_val_accs):.5f}')
print(f'  F1 Score      : {stack_f1:.5f}')
print(f'  Threshold     : {optimal_thr:.4f}')

if stack_acc >= 0.95:
    print('✓ Target Accuracy ≥ 0.95 ACHIEVED on train')
else:
    print(f'⚠ Train accuracy {stack_acc:.5f} below 0.95 — model tuning applied')

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Stacking Ensemble (OOF)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Threshold vs Accuracy curve
thresholds = np.linspace(0.1, 0.9, 161)
accs = [accuracy_score(y_train, (stack_proba >= t).astype(int)) for t in thresholds]

plt.figure(figsize=(10, 4))
plt.plot(thresholds, accs, color='steelblue')
plt.axvline(optimal_thr, color='red', linestyle='--', label=f'Optimal thr={optimal_thr:.3f}')
plt.xlabel('Threshold')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Classification Threshold')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'threshold_curve.png'), dpi=120, bbox_inches='tight')
plt.show()

## 14. SHAP Interpretability

In [ ]:
# Use last-fold XGBoost model
xgb_model = trained_models['XGBoost'][-1]
shap_sample = X_train.sample(min(2000, len(X_train)), random_state=RANDOM_STATE)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(shap_sample)

if isinstance(shap_values, list):
    shap_values = shap_values[1]

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, shap_sample, show=False, max_display=20)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'shap_summary.png'), dpi=120, bbox_inches='tight')
plt.show()
print('✓ SHAP summary plot saved')

In [ ]:
# Top features by mean |SHAP|
mean_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'feature': shap_sample.columns, 'mean_abs_shap': mean_shap})
shap_df = shap_df.sort_values('mean_abs_shap', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=shap_df, x='mean_abs_shap', y='feature', palette='viridis_r')
plt.title('Top 15 Features by Mean |SHAP|')
plt.xlabel('Mean |SHAP Value|')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'shap_feature_importance.png'), dpi=120, bbox_inches='tight')
plt.show()

print('Top 10 features:')
print(shap_df.head(10).to_string(index=False))

## 15. Generate Submission

In [ ]:
# Test predictions
test_preds = predict_test(trained_models, X_test)
test_stack_proba = meta_lr.predict_proba(test_preds)[:, 1]
test_labels = (test_stack_proba >= optimal_thr).astype(int)

submission = pd.DataFrame({'id': test_raw['id'].values, 'Churn': test_labels})
submission_path = os.path.join(RESULTS_DIR, 'submission.csv')
submission.to_csv(submission_path, index=False)

# Verify format
assert list(submission.columns) == list(sample_sub.columns), 'Column mismatch!'
assert len(submission) == len(sample_sub), 'Row count mismatch!'

print(f'✓ Submission saved: {submission_path}')
print(f'  Rows: {len(submission):,}')
print(f'  Churn rate: {test_labels.mean():.4f}')
submission.head()

In [ ]:
# Save pipeline artifacts
pipeline = {
    'trained_models': trained_models,
    'meta_lr': meta_lr,
    'threshold': optimal_thr,
    'artifacts': artifacts,
    'scaler': scaler,
}
joblib.dump(pipeline, os.path.join(MODELS_DIR, 'pipeline.joblib'))
print(f'✓ Pipeline saved → {MODELS_DIR}/pipeline.joblib')

In [ ]:
# Final metrics summary
print('='*55)
print('FINAL RESULTS SUMMARY')
print('='*55)
print(f'Baseline LR AUC      : {lr_scores.mean():.5f}')
for name, scores in cv_results.items():
    print(f'{name:15s} AUC  : {scores.mean():.5f}')
print(f'Stacking OOF AUC     : {stack_auc:.5f}')
print(f'Train Accuracy (OOF) : {stack_acc:.5f}')
print(f'Val Accuracy (OOF)   : {mean_val_acc:.5f}')
print(f'OOF F1 Score         : {stack_f1:.5f}')
print(f'Optimal Threshold    : {optimal_thr:.4f}')
print('='*55)

if stack_auc >= 0.90:
    print('✓ Target AUC ≥ 0.90 ACHIEVED')
else:
    print(f'⚠ AUC target 0.90 not reached (got {stack_auc:.5f})')

if stack_acc >= 0.95:
    print('✓ Target Accuracy ≥ 0.95 ACHIEVED on train')
else:
    print(f'⚠ Train accuracy target 0.95 not reached (got {stack_acc:.5f})')

if mean_val_acc >= 0.95:
    print('✓ Target Accuracy ≥ 0.95 ACHIEVED on val')
else:
    print(f'⚠ Val accuracy target 0.95 not reached (got {mean_val_acc:.5f})')
